# Setup
| Component | Configuration |
|---|---|
| Base model | `distilbert/distilroberta-base` |
| Dataset | `fancyzhx/ag_news` |
| Model class | `AutoModelForSequenceClassification` |
| Trainer | Transformers `Trainer` |
| Fine-tuning method | **Full fine-tuning (no LoRA/PEFT)** |
| Trainable modules | Entire encoder and classification head |
| Evaluation | Accuracy and macro F1 |
| Max sequence length | 256 |
| Training | 3 epochs, batch size 8, accumulation 4 |
| Optimizer | AdamW, linear schedule, 6% warm-up |

This notebook follows the **same cell order and training setup** as `cs420-seminar.ipynb`. It trains all parameters rather than using LoRA.

**Learning rate:** `2e-5` instead of the reference LoRA learning rate (`2e-4`). Full fine-tuning generally needs a smaller learning rate. Other training settings are preserved. The test set is used only after checkpoint selection based on validation macro-F1.


In [ ]:
%pip install -q -U transformers datasets accelerate scikit-learn google-cloud-bigquery-storage pyarrow==23.0.1


In [ ]:
%pip uninstall -y torchao

In [ ]:
import subprocess
import sys


subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=True,
)

check_code = """
import importlib.util
from importlib.metadata import PackageNotFoundError, version

try:
    print("Installed torchao version:", version("torchao"))
except PackageNotFoundError:
    print("torchao package metadata: absent")

print("torchao import location:", importlib.util.find_spec("torchao"))
"""

result = subprocess.run(
    [sys.executable, "-c", check_code],
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)

In [ ]:
import os

# Use one GPU, including when Kaggle provides two T4 GPUs.
# Run this before importing torch in a fresh session.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path

import numpy as np
import torch
from transformers import set_seed

# Both the model and AG News dataset are public. HF_TOKEN is optional.
try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None

MODEL_ID = "distilbert/distilroberta-base"
DATASET_ID = "fancyzhx/ag_news"

OUTPUT_DIR = Path("/kaggle/working/distilroberta-agnews")
MODEL_DIR = OUTPUT_DIR / "final-model"

SEED = 42
MAX_LENGTH = 256
SMOKE_TEST = False

set_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings.")

USE_BF16 = torch.cuda.is_bf16_supported()

print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", USE_BF16)
print("Hugging Face token available:", HF_TOKEN is not None)


In [ ]:
from datasets import DatasetDict, load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding


raw = load_dataset(
    DATASET_ID,
    token=HF_TOKEN,
)

label_names = raw["train"].features["label"].names

id2label = {
    index: name
    for index, name in enumerate(label_names)
}
label2id = {
    name: index
    for index, name in id2label.items()
}

# Create validation data from the official training split.
split = raw["train"].train_test_split(
    test_size=0.1,
    stratify_by_column="label",
    seed=SEED,
)

datasets = DatasetDict(
    {
        "train": split["train"],
        "validation": split["test"],
        "test": raw["test"],
    }
)

if SMOKE_TEST:
    datasets["train"] = (
        datasets["train"]
        .shuffle(seed=SEED)
        .select(range(512))
    )
    datasets["validation"] = (
        datasets["validation"]
        .shuffle(seed=SEED)
        .select(range(256))
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
)


def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized = datasets.map(
    tokenize,
    batched=True,
    remove_columns=["text"],
)

tokenized = tokenized.rename_column("label", "labels")

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)

print("Labels:", id2label)
print(tokenized)

In [ ]:
from transformers import AutoModelForSequenceClassification


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

# Standard supervised full fine-tuning: train the entire encoder + head.
# No LoRA, PEFT, adapters, or frozen backbone.
for parameter in model.parameters():
    parameter.requires_grad_(True)

total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

assert total_parameters == trainable_parameters
print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Trainable percentage: {100 * trainable_parameters / total_parameters:.2f}%")


In [ ]:
import math

from transformers import TrainingArguments

# 2e-5 for full fine-tuning (the source LoRA run uses 2e-4).


NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4

# Calculate optimizer steps for the single-GPU setup.
batches_per_epoch = math.ceil(
    len(tokenized["train"]) / TRAIN_BATCH_SIZE
)
updates_per_epoch = math.ceil(
    batches_per_epoch / GRADIENT_ACCUMULATION_STEPS
)

total_steps = (
    10
    if SMOKE_TEST
    else updates_per_epoch * NUM_EPOCHS
)

# Warm up over approximately 6% of optimizer steps.
warmup_steps = math.ceil(total_steps * 0.06)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    max_steps=10 if SMOKE_TEST else -1,
    learning_rate=2e-5,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    bf16=USE_BF16,
    fp16=not USE_BF16,
    logging_steps=1 if SMOKE_TEST else 50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    push_to_hub=False,
    label_names=["labels"],
    seed=SEED,
    data_seed=SEED,
)

print("Total optimizer steps:", total_steps)
print("Warm-up steps:", warmup_steps)


In [ ]:
import time

import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer


def compute_metrics(eval_prediction):
    logits = eval_prediction.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=-1)
    labels = eval_prediction.label_ids

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
    }


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Track peak PyTorch CUDA memory during training (not the entire notebook).
device = torch.cuda.current_device()
torch.cuda.synchronize(device)
torch.cuda.reset_peak_memory_stats(device)

start_time = time.perf_counter()
train_result = trainer.train()
torch.cuda.synchronize(device)
wall_time_seconds = time.perf_counter() - start_time

bytes_per_gib = 1024 ** 3
resource_metrics = {
    "wall_time_seconds": round(wall_time_seconds, 2),
    "wall_time_minutes": round(wall_time_seconds / 60, 2),
    "gpu_peak_allocated_gib": round(
        torch.cuda.max_memory_allocated(device) / bytes_per_gib, 3
    ),
    "gpu_peak_reserved_gib": round(
        torch.cuda.max_memory_reserved(device) / bytes_per_gib, 3
    ),
    "gpu_current_allocated_gib": round(
        torch.cuda.memory_allocated(device) / bytes_per_gib, 3
    ),
}

train_result.metrics.update(resource_metrics)
print("\n========== TRAINING RESOURCE SUMMARY ==========")
for key, value in resource_metrics.items():
    print(f"{key}: {value}")
print("===============================================\n")


In [ ]:
from pathlib import Path

import json
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report


RESULTS_DIR = Path("/kaggle/working/output/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Save the complete fine-tuned model, classifier, and tokenizer.
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
trainer.model.config.save_pretrained(str(MODEL_DIR))

# The same metrics CSV convention as the reference notebook, plus
# training-time and memory columns for full fine-tuning comparisons.
train_metrics_df = pd.DataFrame([{
    "model": MODEL_ID,
    "fine_tuning": "full",
    "trainable_parameters": trainable_parameters,
    "total_parameters": total_parameters,
    **train_result.metrics,
}])
train_metrics_df.to_csv(RESULTS_DIR / "training_metrics.csv", index=False)

history_df = pd.DataFrame(trainer.state.log_history)
history_df.to_csv(RESULTS_DIR / "training_history.csv", index=False)

if SMOKE_TEST:
    print("Smoke test completed. Official test evaluation skipped.")
else:
    result = trainer.predict(
        tokenized["test"],
        metric_key_prefix="test",
    )

    logits = result.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    predicted_ids = np.argmax(logits, axis=-1)
    true_ids = result.label_ids

    probabilities = torch.softmax(
        torch.as_tensor(logits, dtype=torch.float32),
        dim=-1,
    ).numpy()

    # Overall evaluation metrics: one row; append resource metrics.
    metrics_df = pd.DataFrame([{
        "model": MODEL_ID,
        "fine_tuning": "full",
        "trainable_parameters": trainable_parameters,
        "total_parameters": total_parameters,
        **result.metrics,
        **resource_metrics,
    }])
    metrics_df.to_csv(
        RESULTS_DIR / "metrics.csv",
        index=False,
    )

    # Precision, recall, F1, and support for each class.
    report = classification_report(
        true_ids,
        predicted_ids,
        labels=list(range(len(label_names))),
        target_names=label_names,
        output_dict=True,
        zero_division=0,
    )

    report_rows = []

    for category, scores in report.items():
        if isinstance(scores, dict):
            report_rows.append(
                {
                    "category": category,
                    **scores,
                }
            )

    report_df = pd.DataFrame(report_rows)
    report_df.to_csv(
        RESULTS_DIR / "classification_report.csv",
        index=False,
    )

    # Predictions follow the original test dataset order.
    predictions_df = pd.DataFrame(
        {
            "test_index": np.arange(len(true_ids)),
            "text": datasets["test"]["text"],
            "true_label_id": true_ids,
            "true_label": [id2label[int(i)] for i in true_ids],
            "predicted_label_id": predicted_ids,
            "predicted_label": [
                id2label[int(i)] for i in predicted_ids
            ],
            "correct": true_ids == predicted_ids,
        }
    )

    for class_id, class_name in id2label.items():
        predictions_df[f"probability_{class_name}"] = (
            probabilities[:, class_id]
        )

    predictions_df.to_csv(
        RESULTS_DIR / "predictions.csv",
        index=False,
    )

    display(metrics_df)
    display(report_df)

    print(f"CSV results saved to: {RESULTS_DIR}")


In [ ]:
device = next(trainer.model.parameters()).device


def classify_news(text):
    inputs = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)

    trainer.model.eval()

    with torch.inference_mode():
        logits = trainer.model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_id = probabilities.argmax().item()

    return {
        "label": id2label[predicted_id],
        "scores": {
            id2label[index]: float(score)
            for index, score in enumerate(probabilities.cpu().tolist())
        },
    }


classify_news(
    "The team scored twice in the final minutes to win the championship."
)

In [ ]:
import shutil

from IPython.display import FileLink, display


archive_path = shutil.make_archive(
    "/kaggle/working/distilroberta-agnews-full-model",
    "zip",
    root_dir=str(MODEL_DIR),
)

display(FileLink(archive_path))
